# Exploratory Data Analysis — Real 2025 Flights

Charts use the Unity Catalog v1 curated table, not toy data.

In [ ]:

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

schema = "dsai3202_databricks.proj_flight60106354"
real_eda = spark.table(f"{schema}.flights_features_2025")

class_counts = {
    int(r["ArrDel15"]): r["count"]
    for r in real_eda.groupBy("ArrDel15").count().collect()
}

on_time = class_counts.get(0, 0)
delayed = class_counts.get(1, 0)

plt.figure(figsize=(8, 5))
bars = plt.bar(["On Time", "Delayed"], [on_time, delayed])

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center", va="bottom"
    )

plt.title("2025 Flight Delay Distribution - Real Data")
plt.ylabel("Number of Flights")
plt.tight_layout()
plt.show()

print("Delayed flights:", delayed)
print("On-time flights:", on_time)


In [ ]:

airline_stats = (
    real_eda.groupBy("Reporting_Airline")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("DelayRate")
    )
    .orderBy(F.desc("TotalFlights"))
)

top_airlines = airline_stats.limit(10).toPandas()

plt.figure(figsize=(10, 5))
plt.bar(
    top_airlines["Reporting_Airline"],
    top_airlines["DelayRate"]
)
plt.title("2025 Delay Rate - 10 Busiest Airlines")
plt.xlabel("Airline")
plt.ylabel("Delay Rate (%)")
plt.tight_layout()
plt.show()

display(airline_stats)


In [ ]:

hour_stats = (
    real_eda.groupBy("DepHour")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("DelayRate")
    )
    .orderBy("DepHour")
)

hour_pd = hour_stats.toPandas()

plt.figure(figsize=(10, 5))
plt.plot(
    hour_pd["DepHour"],
    hour_pd["DelayRate"],
    marker="o"
)
plt.title("2025 Flight Delay Rate by Scheduled Departure Hour")
plt.xlabel("Scheduled Departure Hour")
plt.ylabel("Delay Rate (%)")
plt.xticks(range(0, 24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(hour_stats)
